In [3]:
# ============================================================
# Cell #1: Stage Connection and Manual Control
# ============================================================

from hardware import RemoteHardwareManager
import numpy as np

hw = RemoteHardwareManager(host="127.0.0.1", port=18861)

# check remote stage
current_pos = hw.stage_get_position()
print(f"📍 Current stage position: {current_pos:.4f} mm")
# get rotating stage angle (deg):
angle = hw.rotation_get_position()
print(f"🔄 Current rotation stage angle: {angle:.2f} °")

# Check remote SLM and upload a fresnel test pattern
from phase_generators import PhaseGenerator
from optics_utils import load_dict_from_json
params = load_dict_from_json(r".\\config\\base.json")
params['M'] = 5
Optimizer = PhaseGenerator(params)
Optimizer.generate(mode='fresnel')
phase_8bit = Optimizer.update_phase_8bit()
hw.upload_slm(phase_8bit)
print(f"✅ Test pattern with M={params.get('M', 'default')} uploaded to SLM.")



# ---- Reference commands (uncomment to use) ----

# Home the stage:
# hw.stage_home()

# Move to absolute position (mm):
# hw.stage_move_to(11.805)
# pos = hw.stage_get_position()
# print(f"Position: {pos:.4f} mm")


# rotate to absolute angle (deg):
M_angle = {'M3': 282.75, 'M5': 268.0, 'M7': 261.6, 'M9': 258.5}

target_angle = M_angle['M5']
hw.rotation_move_to(target_angle+10) # to avoid backlash
hw.rotation_move_to(target_angle)
print(f"🔄 Current rotation stage angle: {hw.rotation_get_position():.2f} °")


Connecting to local hardware service at 127.0.0.1:18861 ...
✅ Connected to unified hardware service
   Stage status:
     - Stage 1 (PRM1-Z8 (Rotation)): ✅ Connected
     - Stage 2 (Z825B (Z-Axis)): ✅ Connected
📍 Current stage position: 12.0050 mm
🔄 Current rotation stage angle: 258.50 °
Using device: cuda
F/47.31, 73.89999999999999 mm
Lens width: 1.562mm
Airy radius: 29.7um
Depth of focus: 2.31mm
Multi-depth planes are used at F=72.7 (-0.5DOF), 75.1 (0.5DOF) mm
Fresnel Lens: 850x850 px; 5x5 lenses
🚀 Phase pattern sent to SLM
✅ Test pattern with M=5 uploaded to SLM.
🔄 Current rotation stage angle: 268.00 °


In [1]:
# ============================================================
# Cell #2: NPY File Selection GUI
# ============================================================
# Scan for .npy files in ./output folder

from npy_file_selector import select_npy_files

file_selector_widget = select_npy_files(output_dir="./output")


✅ Found 24 .npy files in output
Select .npy files for scanning (Ctrl+Click for multiple):
Files without M%d pattern will be automatically excluded.



In [ ]:
# ============================================================
# Cell #3: Z-Scan Acquisition
# ============================================================

from hardware import RemoteHardwareManager
import numpy as np
import time
import os
from datetime import datetime
from nas_mapper import quick_map

# ---- Need map NAS if using NAS path ----
#### Note: use Z: drive for NAS paths!
success, msg = quick_map()
if not success:
    raise RuntimeError(f"❌ NAS mapping failed: {msg}")

# ---- User Parameters ----
# Note: NAS sharing folder \\iOptics-NIR-II\data will be mapped to Z: drive
save_dir = r"Z:\\SLM_super_resolution\\data\\for_auto_scan\\"  
z_focal_plane = 11.805  # Focal plane position in mm

# settings for 5x5 lens array
num_steps = 81  # Number of z positions to scan
z_range = 0.4  # Total range to scan (± around focal plane)

# # settings for 7x7 lens array
# num_steps = 81  # Number of z positions to scan
# z_range = 0.6  # Total range to scan (± around focal plane)

# # settings for 9x9 lens array
# num_steps = 81  # Number of z positions to scan
# z_range = 0.7  # Total range to scan (± around focal plane)

# -----------------------------------


# ---- Get selected .npy files info from previous cell ----
selected_npy_files = file_selector_widget.get_selected_files()
output_dir = file_selector_widget.get_output_dir()
m_list = file_selector_widget.get_m_patterns()

# ---- Calculate z positions ----
z_positions = np.linspace(z_focal_plane - z_range/2, z_focal_plane + z_range/2, num_steps)

# ---- Verify selection ----
if not selected_npy_files:
    raise ValueError("❌ No .npy files selected! Go back to Cell #2 and select files.")

# ---- Connect to hardware ----
hw = RemoteHardwareManager(host="127.0.0.1", port=18861)

# ---- Capture click position ----
print("\n🎯 Capturing camera trigger button position...")
click_pos = hw.capture_position()
if click_pos is None:
    raise RuntimeError("❌ Failed to capture click position!")

# ---- Record scan start time and parameters ----
scan_start_time = datetime.now()
scan_info = {
    'start_time': scan_start_time.isoformat(),
    'z_focal_plane': z_focal_plane,
    'z_range': z_range,
    'num_steps': num_steps,
    'z_positions': z_positions.tolist(),
    'patterns': selected_npy_files,
    'save_dir': save_dir,
}

# ---- Main scanning loop ----
total_frames = len(selected_npy_files) * num_steps
frame_counter = 0


# # pre-trigger camera (when software not activated)
# hw.click_at()
# time.sleep(0.5)

# clear any existing tiff files in temp save_dir
for file in os.listdir(save_dir):
    if file.endswith(".tiff") or file.endswith(".tif"):
        os.remove(os.path.join(save_dir, file))

for pattern_idx, npy_name in enumerate(selected_npy_files):
    m_pattern = m_list[pattern_idx]
    npy_path = os.path.join(output_dir, npy_name)
    
    print(f"\n[Pattern {pattern_idx+1}/{len(selected_npy_files)}] {npy_name} | {m_pattern}")
    
    # Load and upload pattern to SLM 
    pattern = np.load(npy_path)
    hw.upload_slm(pattern)
    print(f"   ✅ Pattern uploaded to SLM")

    # switch rotation stage to corresponding angle if m_pattern changed
    if pattern_idx == 0 or m_pattern != m_list[pattern_idx - 1]:
        target_angle = M_angle[m_pattern]
        hw.rotation_move_to(target_angle+10) # to avoid backlash
        hw.rotation_move_to(target_angle)
        print(f"   🔄 Rotation stage moved to {hw.rotation_get_position():.2f} degrees")
    
    # init stage position
    hw.stage_move_to(z_positions[0])
    time.sleep(0.2)

    # Z-scan sub-loop
    for z_idx, z_pos in enumerate(z_positions):
        frame_counter += 1
        
        # Move stage
        hw.stage_move_to(z_pos)
        time.sleep(0.2)  # Wait for settling
        
        # Trigger camera
        hw.click_at()
        time.sleep(0.65)  # Wait for capture
        
        # Progress display
        progress = frame_counter / total_frames * 100
        print(f"\r   Z[{z_idx+1}/{num_steps}] = {z_pos:.4f} mm | "
              f"Frame {frame_counter}/{total_frames} ({progress:.1f}%)", end='')
    
    print()  # New line after each pattern

scan_end_time = datetime.now()
scan_info['end_time'] = scan_end_time.isoformat()
scan_info['duration_seconds'] = (scan_end_time - scan_start_time).total_seconds()

print(f"\n{'='*60}")
print(f"✅ Scan completed at {scan_end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   Duration: {scan_info['duration_seconds']:.1f} seconds")
print(f"   Total frames captured: {frame_counter}")
print(f"📐 Z-scan range: {z_positions[0]:.4f} mm to {z_positions[-1]:.4f} mm")
print(f"   Step size: {(z_positions[1]-z_positions[0]):.4f} mm")
print(f"   Total frames per pattern: {num_steps}")
print(f"   Total patterns: {len(selected_npy_files)}")
print(f"   Total frames: {num_steps * len(selected_npy_files)}")
print(f"{'='*60}")

Connecting to local hardware service at 127.0.0.1:18861 ...
✅ Connected to unified hardware service

🎯 Capturing camera trigger button position...
📍 Captured position: (361, 525)

[Pattern 1/8] fresnel_M5_F74.5e-3.npy
🚀 Phase pattern sent to SLM
   ✅ Pattern uploaded to SLM
🖱️ Clicked at (361, 525)
   Z[1/81] = 11.6050 mm | Frame 1/648 (0.2%)🖱️ Clicked at (361, 525)
   Z[2/81] = 11.6100 mm | Frame 2/648 (0.3%)🖱️ Clicked at (361, 525)
   Z[3/81] = 11.6150 mm | Frame 3/648 (0.5%)🖱️ Clicked at (361, 525)
   Z[4/81] = 11.6200 mm | Frame 4/648 (0.6%)🖱️ Clicked at (361, 525)
   Z[5/81] = 11.6250 mm | Frame 5/648 (0.8%)🖱️ Clicked at (361, 525)
   Z[6/81] = 11.6300 mm | Frame 6/648 (0.9%)🖱️ Clicked at (361, 525)
   Z[7/81] = 11.6350 mm | Frame 7/648 (1.1%)🖱️ Clicked at (361, 525)
   Z[8/81] = 11.6400 mm | Frame 8/648 (1.2%)🖱️ Clicked at (361, 525)
   Z[9/81] = 11.6450 mm | Frame 9/648 (1.4%)🖱️ Clicked at (361, 525)
   Z[10/81] = 11.6500 mm | Frame 10/648 (1.5%)🖱️ Clicked at (361, 525)
   Z[11/

In [5]:
import os
import glob
import shutil
import json
from datetime import datetime

# ---- User Parameters ----
user_prefix = "PSF_4um"  # Custom prefix for renamed files

# ---- Use scan_info from last Cell ----
# (Assumes scan_info, selected_npy_files, z_positions, save_dir are available)

print(f"📂 Organizing results in: {save_dir}")
print(f"   User prefix: {user_prefix}")

# Find all tiff files generated by the scan
tiff_pattern = os.path.join(save_dir, "ss_single_*.tiff")
tiff_files = sorted(glob.glob(tiff_pattern), 
                    key=lambda x: int(os.path.basename(x).replace('ss_single_', '').replace('.tiff', '')))

expected_frames = len(selected_npy_files) * len(z_positions)
print(f"   Found {len(tiff_files)} tiff files (expected: {expected_frames})")

if len(tiff_files) != expected_frames:
    print(f"⚠️ Warning: Frame count mismatch!")

# ---- Organize files by pattern ----
frame_idx = 0
for pattern_idx, npy_name in enumerate(selected_npy_files):
    # Create subfolder with pattern name (without .npy extension)
    pattern_basename = os.path.splitext(npy_name)[0]
    pattern_folder = os.path.join(save_dir, pattern_basename)
    os.makedirs(pattern_folder, exist_ok=True)
    
    print(f"\n[{pattern_idx+1}/{len(selected_npy_files)}] {pattern_basename}/")
    
    # Move and rename tiff files for this pattern
    for z_idx, z_pos in enumerate(z_positions):
        if frame_idx >= len(tiff_files):
            print(f"   ⚠️ Missing frame {frame_idx+1}")
            frame_idx += 1
            continue
        
        src_path = tiff_files[frame_idx]
        
        # New filename: userprefix_frame%d_z%.4fmm.tiff
        new_name = f"{user_prefix}_frame{z_idx+1:03d}_z{z_pos:.4f}mm.tiff"
        dst_path = os.path.join(pattern_folder, new_name)
        
        # Move and rename
        shutil.move(src_path, dst_path)
        frame_idx += 1
    
    print(f"   ✅ Moved {len(z_positions)} frames")
    
    # ---- Generate scan info file for this pattern ----
    info_filename = f"{pattern_basename}_scan_info.json"
    info_path = os.path.join(pattern_folder, info_filename)
    
    pattern_info = {
        'pattern_name': npy_name,
        'user_prefix': user_prefix,
        'scan_start_time': scan_info['start_time'],
        'scan_end_time': scan_info['end_time'],
        'duration_seconds': scan_info['duration_seconds'],
        'z_focal_plane_mm': scan_info['z_focal_plane'],
        'z_range_mm': scan_info['z_range'],
        'num_steps': scan_info['num_steps'],
        'z_positions_mm': scan_info['z_positions'],
        'z_step_mm': z_positions[1] - z_positions[0] if len(z_positions) > 1 else 0,
        'file_naming': f"{user_prefix}_frame{{frame}}_z{{z:.4f}}mm.tiff",
        'total_frames': len(z_positions),
    }
    
    with open(info_path, 'w') as f:
        json.dump(pattern_info, f, indent=2)
    
    print(f"   ✅ Saved {info_filename}")

# ---- Generate master scan log ----
master_log_path = os.path.join(save_dir, f"scan_log_{scan_start_time.strftime('%Y%m%d_%H%M%S')}.json")
master_info = {
    'scan_start_time': scan_info['start_time'],
    'scan_end_time': scan_info['end_time'],
    'duration_seconds': scan_info['duration_seconds'],
    'user_prefix': user_prefix,
    'z_focal_plane_mm': scan_info['z_focal_plane'],
    'z_range_mm': scan_info['z_range'],
    'num_steps': scan_info['num_steps'],
    'z_positions_mm': scan_info['z_positions'],
    'patterns': selected_npy_files,
    'total_patterns': len(selected_npy_files),
    'total_frames': expected_frames,
    'output_folders': [os.path.splitext(f)[0] for f in selected_npy_files],
}

with open(master_log_path, 'w') as f:
    json.dump(master_info, f, indent=2)

print(f"\n{'='*60}")
print(f"✅ Organization complete!")
print(f"   Master log: {os.path.basename(master_log_path)}")
print(f"   Created {len(selected_npy_files)} pattern folders")
print(f"{'='*60}")

📂 Organizing results in: Z:\\SLM_super_resolution\\data\\for_auto_scan\\
   User prefix: PSF_4um
   Found 648 tiff files (expected: 648)

[1/8] fresnel_M5_F74.5e-3/
   ✅ Moved 81 frames
   ✅ Saved fresnel_M5_F74.5e-3_scan_info.json

[2/8] opt7_M5_over0.5_airy1_mask0/
   ✅ Moved 81 frames
   ✅ Saved opt7_M5_over0.5_airy1_mask0_scan_info.json

[3/8] opt8_M5_over0.6_airy1_mask0/
   ✅ Moved 81 frames
   ✅ Saved opt8_M5_over0.6_airy1_mask0_scan_info.json

[4/8] opt9_M5_over0.7_airy1_mask0/
   ✅ Moved 81 frames
   ✅ Saved opt9_M5_over0.7_airy1_mask0_scan_info.json

[5/8] opt10_M5_over0.8_airy1_mask0/
   ✅ Moved 81 frames
   ✅ Saved opt10_M5_over0.8_airy1_mask0_scan_info.json

[6/8] opt11_M5_over0.9_airy1_mask0/
   ✅ Moved 81 frames
   ✅ Saved opt11_M5_over0.9_airy1_mask0_scan_info.json

[7/8] opt18_M5_edof1/
   ✅ Moved 81 frames
   ✅ Saved opt18_M5_edof1_scan_info.json

[8/8] opt19_M5_edof2/
   ✅ Moved 81 frames
   ✅ Saved opt19_M5_edof2_scan_info.json

✅ Organization complete!
   Master log